  ## Guardrails with LangChain — Crash Course

This notebook covers everything you need to know about implementing Guardrails in LangChain agents using the middleware system.

📚 Topics Covered
1. What are Guardrails & Why do they matter?
2. Two approaches: Deterministic vs Model-based
3. Built-in: PII Detection Middleware
4. Built-in: Human-in-the-Loop Middleware
5. Custom: Before-Agent Guardrail (input filtering)
6. Custom: After-Agent Guardrail (output safety)
7. Layered / Combined Guardrails
8. Real-World Use Case: Healthcare Chatbot

### 📌 Docs Reference: https://docs.langchain.com/oss/python/langchain/guardrails

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Setup API Key

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")

## 🧠 Section 1: What are Guardrails?
Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

- Before the agent starts (input guardrails)
- After it completes (output guardrails)
- Around model and tool calls




### Common Use Cases:
       Use Case	                                            Example
1. PII leakage prevention	                            Redact emails/credit cards before logging

2. Prompt injection blocking	                        Detect adversarial inputs

3. Harmful content filtering	                        Block dangerous requests

4. Business rule enforcement	                        Require approval for financial ops

5. Output quality validation	                        Ensure response meets safety standards



## ⚖️ Section 2: Two Approaches to Guardrails
Deterministic Guardrails

- Rule-based: regex, keyword matching, explicit checks

            ✅ Fast, predictable, cost-effective

❌ May miss nuanced violations

- Model-Based Guardrails

         Uses LLMs/classifiers for semantic understanding

         ✅ Catches subtle/nuanced issues

         ❌ Slower and more expensive

In [3]:
# Quick Illustration of the two approaches

#------ Determinstic approach---- 

def determinstic_guardrail(text:str)-> bool:
    """Returns True if content is blocked   """
    banned_keywords= ["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs= ["How do I hack into a database?", 
            "What is the capital of France?", 
            "Explain how malware spreads",]

print("=== Determinstic Guardrail Demo ===")

for inp in test_inputs:
    blocked = determinstic_guardrail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")

=== Determinstic Guardrail Demo ===
Blocked:How do I hack into a database?
Allowed:What is the capital of France?
Blocked:Explain how malware spreads


In [4]:
from langchain_groq import ChatGroq

def model_based_guradrail(text:str)-> str: 
    """uses an LLM to evaluate content safety, Returns SAFE or UNSAFE. """

    model= ChatGroq(model="qwen/qwen3.8-27b", temperature= 0)
    prompt= f""" Is the following user input safe to process? 
    Reply with only 'SAFE' or UNSAFE'. 

    Input:{text}"""

    result= model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("==== Model- based Guardrail Demo=== ")

for inp in test_inputs:
    verdict= model_based_guradrail(inp)
    status= " UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

==== Model- based Guardrail Demo=== 
 UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads


## 🔒 Section 3: Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

- Supported PII Types:
    Type	Example
    email	user@example.com

    credit_card	5105-1051-0510-5100

    ip	192.168.1.1

    mac_address	00:1A:2B:3C:4D:5E

    url	https://secret-site.com

- Strategies:
    Strategy	Result
    redact	[REDACTED_EMAIL]

    mask	****-****-****-1234

    hash	a8f5f167...
    
    block	Raises an exception


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

@tool
def customer_lookup(query:str)->str:
    """ Look up customer information.   """
    return f" Customer record found for query: {query}"

model= ChatGroq(model="qwen/qwen3.8-27b")

agent= create_agent(
    model= model,
    tools= [customer_lookup],
    middleware=[ 
        #Readct emails in user input before sending to model
        PIIMiddleware("email", 
                    strategy= "redact", 
                    apply_to_input= True,), 

        #Mask Credit Cards in user input
        PIIMiddleware(
            "credit_card",
            strategy= "mask",
            apply_to_input= True
        ),

        # Block API Keys- raise error if detetcted 

        PIIMiddleware("api_key", 
                    detector = r"sk- [a-zA-Z0-9]{32}",
                    strategy= "block", 
                    apply_to_input= True,), 

    ],
)

print("Agent with PII middleware created successfully !")

Agent with PII middleware created successfully !


In [6]:
## Test PII Redaction

result = agent.invoke({
    "messages": [{ "role" :"user", 
                    "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me? "

    }]
})

print("====Agent Response===")
print(result["messages"][-1].content)

====Agent Response===
I can help, but I don't have access to your email or card details in this conversation, and I shouldn't ask you to share sensitive information like your full card number or email address here.

Could you let me know what you'd like help with? For example:
- A question about an account or transaction
- Troubleshooting an issue
- General guidance

If I need to look up your account, I'll only need the minimum identifying information required, and I'll avoid asking for full card numbers or passwords.


In [7]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me? ', additional_kwargs={}, response_metadata={}, id='3b690933-1974-48e8-84f3-b6c2c9e4d2a0'),
  AIMessage(content="I can help, but I don't have access to your email or card details in this conversation, and I shouldn't ask you to share sensitive information like your full card number or email address here.\n\nCould you let me know what you'd like help with? For example:\n- A question about an account or transaction\n- Troubleshooting an issue\n- General guidance\n\nIf I need to look up your account, I'll only need the minimum identifying information required, and I'll avoid asking for full card numbers or passwords.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 298, 'total_tokens': 407, 'completion_time': 0.217800728, 'completion_tokens_details': None, 'prompt_time': 0.021662674, 'prompt_tokens_details': None, 'queue_

In [8]:
## Test API Key Blocking

try: 
    result = agent.invoke({"messages":[{
        "role":"user",
        "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456 "
    }]
    
})
except Exception as e:
    print(f" Blocekd as expected:{e}")

## 👤 Section 4: Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

Best for:

- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

Key requirement: A checkpointer for state persistence across interrupts.

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str)-> str: 
    """ Search the web for information """
    return f"Search results for :{query}"


@tool
def send_email(to:str, subject:str, body:str)-> str:
    """ Send an email to a recipient.    """
    return f"Email sent to {to} with subjects:{subject}"

@tool
def delete_records(table: str, condiiton:str)->str:
    """ Delete records from the database. """

model= ChatGroq(model="qwen/qwen3.8-27b")
## create agent with HITL middleware
hitl_agent= create_agent(
    model=model,
    tools=[search_web,send_email,delete_records],
    middleware= [HumanInTheLoopMiddleware(
        interrupt_on= {
            "send_email": True,  #require approval
            "delete_records": True, #require approval
            "search_web":False, # Auto-approve

        }
    ),
    ],
    checkpointer =InMemorySaver(), #Required for state persistence 
)

print("Human-in-the-loop agent created")


Human-in-the-loop agent created


In [10]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='487c9e82-e0bb-4517-a289-42c14960bb71'), AIMessage(content="I'd be happy to help you send an email about the Q4 results. However, I need more information to compose a meaningful email:\n\n1. **What are the key Q4 results?** (e.g., revenue figures, growth rates, profit margins, specific achievements)\n2. **What is the tone?** (formal, celebratory, cautious, etc.)\n3. **Are there any specific points or highlights** you'd like to emphasize?\n4. **Is there a call to action** or next steps for the team?\n\nIf you have a draft or the specific data you'd like included, please share it and I'll compose the email for you to review before sending.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 435, 'total_tokens': 579, 'completion_time': 0.287668985, 'comple

In [11]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
I'd be happy to help you send an email about the Q4 results. However, I need more information to compose a meaningful email:

1. **What are the key Q4 results?** (e.g., revenue figures, growth rates, profit margins, specific achievements)
2. **What is the tone?** (formal, celebratory, cautious, etc.)
3. **Are there any specific points or highlights** you'd like to emphasize?
4. **Is there a call to action** or next steps for the team?

If you have a draft or the specific data you'd like included, please share it and I'll compose the email for you to review before sending.


In [12]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I attempted to delete all records from the `users` table where `active = false`, but the tool call was rejected.

Would you like me to:
1. Try a different approach or condition?
2. First preview which records would be affected?
3. Something else entirely?

Let me know how you'd like to proceed.


## Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)
 
Use before_agent() to validate or block requests before any LLM processing begins.

## Best for:

- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [16]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

In [17]:

class ContentFilterMiddleware(AgentMiddleware):
    """  Determinstic guardrail: Block requests containing banned keywords. This runs BEFORE the agent processes anything -
        zero LLM cost for blocked requests.   
   """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords =[kw.lower() for kw in banned_keywords]
    
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None
        
        first_message = state["messages"][0]
        if first_message.type!= "human": 
            return None 
        
        content= first_message.content.lower()

        for keyword in self.banned_keywords: 
            if keyword in content:
                print(f" Blocked- keyword detected: '{keyword}'")
                return{
                    "messages":[{
                        "role": "assistant",
                        "content": ("I cannor process requests containing inappropraite content. "
                        "Please repharase your request")
                    }],
                    "jump_to":"end"
                }
        return None

@tool 
def search_tool(query: str)-> str:
    """ Search for Information  """
    return f"Results for: {query}"

model= ChatGroq(model="qwen/qwen3.8-27b")

#create agent with content filter
filtered_agent= create_agent(
    model= model,
    tools= [search_tool],
    middleware= [ContentFilterMiddleware(banned_keywords=["hack","exploit","malware","jailbreak","bypass"]
    ),
    ],
)
print("Content filter created")


Content filter created


In [18]:
# Test 1: Safe Request= Should pass through

result= filtered_agent.invoke({
    "messages":[{"role":"user","content": "what is machine learning?"}]   
})
print("Safe request response:")
print(result["messages"][-1].content)

Safe request response:
Machine learning is a branch of artificial intelligence (AI) that focuses on building systems that can learn from data and improve their performance on tasks over time, without being explicitly programmed for every specific rule.

Here’s a simple breakdown:

- **Traditional programming**: You give a computer a set of explicit rules and data, and it produces an output.
- **Machine learning**: You give the computer data and examples of desired outputs, and it figures out the rules (patterns) on its own to produce the output.

### Common Types of Machine Learning:
1. **Supervised Learning**: The algorithm learns from labeled data (e.g., classifying emails as spam or not spam).
2. **Unsupervised Learning**: The algorithm finds hidden patterns in unlabeled data (e.g., grouping customers by purchasing behavior).
3. **Reinforcement Learning**: The algorithm learns by interacting with an environment and receiving rewards or penalties (e.g., training a robot to walk).

##

In [19]:
# Test 2: UnSafe Request= Should be blocked

result= filtered_agent.invoke({
    "messages":[{"role":"user","content": "How do I hack into a server?"}]   
})
print("Unsafe request response:")
print(result["messages"][-1].content)

 Blocked- keyword detected: 'hack'
Unsafe request response:
I cannor process requests containing inappropraite content. Please repharase your request


## 🔍 Section 6: Custom Guardrail — After-Agent Hook (Output Safety)

Use after_agent() to validate the final agent response before the user sees it.

## Best for:

- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [20]:
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

model= ChatGroq(model="qwen/qwen3.8-27b")

safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [21]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

---
## 🧱 Section 7: Layered / Combined Guardrails

Stack multiple guardrails in the `middleware=[]` array. They execute **in order**, building layered protection.

```
User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response
```

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack

model= ChatGroq(model="qwen/qwen3.8-27b")
production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!
